In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tapas_gmm_modified.policy.models.tpgmm import (
    ReconstructionStrategy, 
    ModelType,
    FittingStage,
    InitStrategy,
    TPGMMConfig,
    AutoTPGMMConfig,
    AutoTPGMM,
    FrameSelectionConfig,
    DemoSegmentationConfig,
    CascadeConfig,
)

from tapas_gmm_modified.dataset.demos import Demos
from tapas_gmm_modified.viz.gmm import plot_hmm_transition_matrix

np.set_printoptions(precision=2)

plt.style.use('default')

from matplotlib import rc
rc('animation', html='jshtml')

In [ ]:
from pathlib import Path

import h5py
from heca.scenes.scene import Scene
from heca.scenes.ogbench.scene import OGBenchScene

from heca.agents.experts.expert import ExpertAgent
from heca.agents.experts.tapas import TapasAgent

scene_cfg = OGBenchScene.Config()
agent_cfg = TapasAgent.Config(
    folder="move_ee",
    scene=scene_cfg,
    use_gt=True,
)
scene = Scene.get(scene_cfg, load=False)
agent = TapasAgent.get(agent_cfg, load=False)
agent_dir = ExpertAgent.resolve(agent_cfg)
model_file_name = "policy_gt.pt" if agent_cfg.use_gt else "policy_img.pt"
file_path = Path.cwd().parent.parent.parent / agent_dir / "demos_post_new.h5"
save_path = Path.cwd().parent.parent.parent / agent_dir / model_file_name
print(agent_dir)
print(file_path)
print(save_path)

In [ ]:
from tensordict import TensorDict
from tapas_gmm_modified.utils.observation import SceneObservation

demos_file = h5py.File(file_path, "r")
# load observations here
print(demos_file.keys())
observations: list[SceneObservation] = []  # type: ignore

demos_scenes, demos_images = scene.load_dataset(demos_file)

for i, (demo_scenes, demo_images) in enumerate(zip(demos_scenes, demos_images)):
    if agent_cfg.use_gt:
        obss: list[TensorDict] = []
        for td_scene in demo_scenes:
            td_obs = td_scene
            td_goal = demo_scenes[-1]
            obs = agent.tapas_td(td_obs, td_goal)
            obss.append(obs)
        stacked_obs = TensorDict.stack(obss, dim=0)
        observations.append(stacked_obs)
    else:
        raise NotImplementedError(
            "TODO: implement tapas_td and convert demos to tapas format"
        )
        # TODO:

In [ ]:
data_kwargs = dict(
    add_init_ee_pose_as_frame=True,
    add_world_frame=False,
    frames_from_keypoints=False,
    kp_indeces=None,
    enforce_z_up=False,
    modulo_object_z_rotation=False,
    make_quats_continuous=True,
)

demos = Demos(observations, **data_kwargs) #type: ignore
print("n_trajs", demos.n_trajs)
print("n_frames", demos.n_frames)
demos.frame_names

In [ ]:
frame_selection_config = FrameSelectionConfig(
    init_strategy=InitStrategy.TIME_BASED,
    fitting_actions=(FittingStage.INIT,),
    use_bic=False,
    drop_redundant_frames=False,
    rel_score_threshold=0.0,
    gt_frames=[[0, 7]],  # Frames per segment
)

tpgmm_config = TPGMMConfig(
    n_components=20,
    model_type=ModelType.HMM,
    use_riemann=True,
    add_time_component=True,
    add_action_component=False,
    position_only=False,
    add_gripper_action=False,
    reg_shrink=1e-2,
    reg_diag=2e-4,
    reg_diag_gripper=2e-2,
    reg_em_finish_shrink=1e-2,
    reg_em_finish_diag=2e-4,
    reg_em_finish_diag_gripper=2e-2,
    trans_cov_mask_t_pos_corr=False,
    em_steps=50,
    fix_first_component=False,  # True maybe
    fix_last_component=False,  # True maybe
    reg_init_diag=5e-4,  # 5
    heal_time_variance=False,
)

demos_segmentation_config = DemoSegmentationConfig(
    gripper_based=False,
    distance_based=False,
    velocity_based=True,
    repeat_final_step=0, #1
    repeat_first_step=0,
    components_prop_to_len=True,
    velocity_threshold=0.1,
)

cascade_config = CascadeConfig(
    kl_keep_time_dim=True,
    kl_keep_rotation_dim=False,
)

auto_tpgmm_config = AutoTPGMMConfig(
    tpgmm=tpgmm_config,
    frame_selection=frame_selection_config,
    demos_segmentation=demos_segmentation_config,
    cascade=cascade_config,
)

In [ ]:
atpgmm = AutoTPGMM(auto_tpgmm_config)


In [ ]:
atpgmm.fit_trajectories(demos, fix_frames=True,
                       init_strategy=InitStrategy.TIME_BASED,
                       fitting_actions=(FittingStage.INIT,)) # FittingStage.EM_HMM))


In [ ]:
atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=True,
    annotate_gaussians=False, annotate_trajs=True,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False) #, size=(150, 10))


In [ ]:
atpgmm.fit_trajectories(demos, fix_frames=True,
                       fitting_actions=(FittingStage.EM_HMM, ))


In [ ]:

atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=True,
    annotate_gaussians=False, annotate_trajs=False,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False)

In [ ]:
atpgmm.plot_model(
    scatter=True, rotations_raw=True, plot_traj=True, plot_gaussians=False,
    annotate_gaussians=True, annotate_trajs=False,
    mean_as_base=False, per_segment=False, gaussian_mean_only=False, plot_traj_means=False, time_based=False)

In [ ]:
atpgmm.plot_hmm_transition_matrix()

In [ ]:
atpgmm.to_disk(save_path) # type: ignore

In [ ]:
seg_local_marginals, seg_trans_marginals, seg_trans_marg_container, seg_joint_models, cascaded_hmms, (reconstructions, original_trajectories, extras) = atpgmm.reconstruct(
    strategy=ReconstructionStrategy.GMR,
    use_ss=False)


In [ ]:
for cascaded_hmm in cascaded_hmms:
    plot_hmm_transition_matrix(cascaded_hmm)

In [ ]:

atpgmm.plot_reconstructions(
    seg_trans_marg_container, cascaded_hmms, reconstructions, original_trajectories,
    plot_trajectories=True, plot_reconstructions=True, plot_gaussians=True,
    time_based=True, equal_aspect=False, per_segment=False)


In [ ]:
atpgmm.plot_reconstructions(
    seg_trans_marginals, seg_joint_models, reconstructions, original_trajectories,
    plot_trajectories=True, plot_reconstructions=True, plot_gaussians=True,
    time_based=False, equal_aspect=True, per_segment=False)
